# Exploratory Data Analysis

- dataset - https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report , confusion_matrix
from sklearn.ensemble import RandomForestClassifier

lets start by importing the dataset and then lets have a look on the columns to understand the data,

In [2]:
diabetes_df = pd.read_csv('/home/spandan/Documents/ML/disease-predictor/data/diabetes.csv')
diabetes_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/home/spandan/Documents/ML/disease-predictor/data/diabetes.csv'

alr lets have look at the columns to get a brief blueprint over the dataset

In [ ]:
diabetes_df.columns

### what are we actually looking at?
seeing the column names is one thing, but we need to know what they actually represent to make any sense of the correlations later. Let's list them out based on the dataset documentation.

#### lets break down and understand the columns first (Given)
1. pregnencies - number of times pregnent
2. Glucose - plasma glucose concentration for a glucose tolorence test
3. BloodPressure - Diastolic Blood Pressure (mm hg)
4. skinThickness - Tricep skin fold thickness (mm)
5. Insulin - 2hr serem insulin
6. BMI - body mass index in kg/m sq
7. DiabetesPedigreeFunction
8. Age
9. Outcomes ( Target Varlaiable ) -- 1 means positive , 0 means negetive for diabetes

In [ ]:
diabetes_df.info()

In [ ]:
diabetes_df.describe()

In [ ]:
diabetes_df['Outcome'].value_counts(normalize=True)

alr , lets check if the dataset contains nulls or 0s for columns which do care for 0s , we will mostly skil the pregnencies , outcomes .

In [ ]:
diabetes_df.isnull().sum()

In [ ]:
(diabetes_df == 0).sum()

here i can see 0s in impossible fields , like glucose can never be zero and if we keep it as it is it will harm the training. similarly for *blood pressure , skinThickness , insulin , BMI*

lets fix the columns ,
common ways to fix include by mean or by median . If we use mean that might get risky for the outliers tho , soo considering a safer option let us replace the 0s with median.
alr let me make a list containing all the cols which needs to be fixed

In [ ]:
cols_for_fixing = [
    'Glucose',
    'BloodPressure',
    'SkinThickness',
    'Insulin',
    'BMI'
]

soo pandas is good with handeling NaN values so lets first repace the zeroes with NaN and then we fill them with median

In [ ]:
diabetes_df[cols_for_fixing] = diabetes_df[cols_for_fixing].replace(0 , np.nan)
diabetes_df.isna().sum()

In [ ]:
(diabetes_df == 0).sum()

### checking my work
just double checking here—the NaNs should have replaced those impossible zeros in our target columns. pregnancies and outcome can stay zero, obviously, but the rest look much cleaner now. time to fill those gaps.

alr that did work , lets fill those with median now.

In [ ]:
for col in cols_for_fixing:
    median = diabetes_df[col].median()
    diabetes_df[col] = diabetes_df[col].fillna(median)

diabetes_df.describe()

the missing values are now handled , next lets vizualize the important features using a heatmap !
## corelation heatmap


In [ ]:
plt.Figure(figsize=(10,8))
sns.heatmap(diabetes_df.corr(), annot=True)
plt.show()

### taking a moment to digest this
okay, looking at that heatmap, the Glucose vs Outcome relationship is screaming at us. it's clearly the heavy hitter here. also, that Age vs Pregnancies link is something we should keep an eye on before we feed this into a linear model.

>the heatmap seems a lot more informative than the dataset
so , we can notice the strongest positive relation wrt
1. Gluecose (thats just directly proportional)
2. BMI , makes sense , in real world senarios as well.

what i can notice is there is multicoliniarity  (string relation between features) at age vs pregnencies (makes sense) , and BMI vs Skin thickness. this can harml logistic regression . still to consider for comparison , lets start with a logistic regreesion Model.

# Logistic Regression
>we are already clear with the dependent and independent vars , Outcome is a dependent var  and rest , lets just take them in y

In [ ]:
x = diabetes_df.drop('Outcome' , axis = 1)
y = diabetes_df['Outcome']

In [ ]:
x

In [ ]:
y

alr then lets split the data for train and test by 80/20 proportion , once done we can move towards scaling

In [ ]:
x_train , x_test , y_train , y_test = train_test_split(x , y , random_state=42 , test_size=0.2)

In [ ]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.fit_transform(x_test)

### training time
scales are set, data is split. let's see how the baseline logistic regression handles this. i'm curious to see if it catches the patterns or gets tripped up by the noise.

In [ ]:
lr_model = LogisticRegression()
lr_model.fit(x_train,y_train)

In [ ]:
lr_pred = lr_model.predict(x_test)

In [ ]:
print(classification_report(y_test,lr_pred))

In [ ]:
cm = confusion_matrix(y_test , lr_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm , annot=True  , cmap = 'Blues')

plt.xlabel("predicted")
plt.ylabel("Actual")
plt.title("Diabetes Logistic Regression")

plt.show()

### Conclusion from Logistic Regression

The Logistic Regression model gave an accuracy of around 77%, which is decent for a medical dataset like this. The model was able to predict non-diabetic patients fairly well, but struggled comparatively more while detecting diabetic patients.

One important observation here is that the recall for diabetic cases is lower than desired. This means the model is still missing a noticeable number of actual diabetic patients (false negatives), which can be risky in real-world medical scenarios.

Another interesting thing noticed during this process was how important preprocessing actually is. Initially, the dataset appeared to have no missing values, but several columns contained medically impossible zeros which were acting as hidden missing values. Replacing them using median imputation improved the dataset quality and made the model training more reliable.

Overall, Logistic Regression works as a solid baseline model for this dataset, but there is still room for improvement using more advanced models like Random Forest.


# Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(random_state=42 , n_estimators=299)
rf_model.fit(x_train , y_train)

In [ ]:
rf_pred = rf_model.predict(x_test)

In [ ]:
print(classification_report(y_test , rf_pred))

In [ ]:
rf_cm = confusion_matrix(y_test , rf_pred)

### how did the forest do?
let's visualize the confusion matrix for the Random Forest. usually, these ensemble models are better at picking up the nuances that simple regression might miss, especially with features like BMI and Insulin acting together.

In [ ]:
sns.heatmap(rf_cm , annot=True)
plt.show()

In [ ]:
importance = pd.DataFrame({
    "features" : x.columns,
    "importance" : rf_model.feature_importances_
})

importance = importance.sort_values(by = "importance" , ascending=False)

print(importance)

In [ ]:
sns.barplot(x = importance['features'] , y = importance['importance'] , data = importance)
plt.show('feature importance')
plt.show()

This dataset was honestly more realistic than the heart disease one. At first it looked clean because there were no null values, but later we found out that many columns had impossible zeros acting as hidden missing values. After handling them using median imputation, the dataset became much cleaner for training.

From the heatmap and feature importance graph, Glucose turned out to be the most important feature for predicting diabetes, followed by BMI and Age. Logistic Regression gave decent baseline performance, while Random Forest slightly improved recall by catching more diabetic patients.

Overall this notebook helped in understanding:

preprocessing and missing value handling
correlation and multicollinearity
Logistic Regression vs Random Forest
confusion matrix and classification metrics
feature importance in tree based models

Compared to the heart dataset, this one felt more noisy and challenging, which made the workflow feel much closer to real world ML problems.